# Diplomatura en ciencia de datos, aprendizaje automático y sus aplicaciones

## Edición 2026 - Optativa: Aprendizaje por Refuerzo

Profesores: Jorge Palombarini y Juan Barsce

---

# Modelos RL offline Decision Transformers (DT) e Implicit Q-Lerning (IQL)

Autor: Gonzalo Perez Paina

Enlaces:
- [Repositorio de la materia](https://github.com/DiploDatos/AprendizajePorRefuerzos/)
- [Consigna del Trabajo Práctico](https://github.com/DiploDatos/AprendizajePorRefuerzos/blob/master/2026/tp/CONSIGNA.md)

## Acceso al dataset y módulos Python base

Las siguientes celdas que permiten trabajar con los datos y módulos Python desde el repositorio de GitHub se elaboraron con ayuda de Claude.

In [ ]:
import os, sys, importlib
REPO = "DiploDatos/AprendizajePorRefuerzos"

In [ ]:
RAIZ = "/content/" + REPO.split("/")[1]          # raíz del repo (clone / pull)
TRABAJO = os.path.join(RAIZ, "2026", "tp")       # directorio de trabajo

if not os.path.exists(RAIZ):
    !git clone -q https://github.com/{REPO}.git {RAIZ}
else:
    !git -C {RAIZ} pull -q

%cd {TRABAJO}
if os.path.exists("requirements.txt"):
    !pip install -q -r requirements.txt
sys.path.insert(0, TRABAJO)                      # para "import protocolo..."

# Parche: el autoreload de IPython en Colab todavía importa 'imp' (eliminado en Python 3.12+)
sys.modules.setdefault("imp", importlib)
%load_ext autoreload
%autoreload 2

/content/AprendizajePorRefuerzos/2026/tp


## Acceso a los scripts de entrenamiento y validación

A continuación se accede a la implementación contenida en un archivo ZIP y se descomprime:

In [ ]:
import urllib.request, zipfile

ZIP_URL = "https://github.com/Mariano-ds/tpRL_2026/raw/refs/heads/main/gonzalo/tp_dt_iql.zip"
ZIP = os.path.join(TRABAJO, "tp_dt_iql.zip")

urllib.request.urlretrieve(ZIP_URL, ZIP)
if not zipfile.is_zipfile(ZIP):          # p. ej. si el repo es privado, GitHub devuelve una página HTML
    raise RuntimeError(f"Lo descargado no es un ZIP: revisar la URL o los permisos del repo\n{ZIP_URL}")

with zipfile.ZipFile(ZIP) as z:
    z.extractall(TRABAJO)                # sobrescribe dt/ e iql/ si ya existían
os.remove(ZIP)

!ls dt iql

dt:
curvas.py    evaluar.py   modelo.py    README.md   repro.py
entrenar.py  __init__.py  politica.py  reporte.py  tests

iql:
entrenar.py  evaluar.py  __init__.py  modelo.py  politica.py  README.md  tests


## Prueba del protocolo

Los números que imprime no significan nada, porque son 1 época y 40 usuarios. Lo único que importa es llegar a TODO OK.

In [ ]:
!python -m experimentos.prueba_corta --dataset netflix

device: cuda | torch 2.14.1+cu130 | dataset netflix

[1/6] Datos y logs
  OK  n_items=752 | usuarios train=14400 val=1600 test=1600 | episodios=73827
  OK  retorno medio de un episodio (random): 64.3   (Netflix en el protocolo: 64,3)
  OK  logs mixto: retorno medio 74.6   (Netflix en el protocolo: 74,6)

[2/6] evaluate_grid y write_reports
  OK  val: 320 sesiones, 4 condiciones, columnas esperadas presentes
  OK  test: 320 sesiones, 4 condiciones, columnas esperadas presentes
  OK  columna de grupo en sesiones: group  (sin ella, el gráfico por grupo sólo muestra el agregado)
  OK  columna de usuario: user_id  (sin ella, el IC 95 % trata cada sesión como unidad)
  OK  métrica hit_rate: columna hit_rate  (sin ella, la tabla la marca como ausente)
  OK  métrica ndcg@10: columna ndcg@10  (sin ella, la tabla la marca como ausente)
  OK  métrica precision@10: columna precision@10  (sin ella, la tabla la marca como ausente)
      columnas de sesiones.csv: ['user_id', 'group', 'seed', 'return'

## Ejecución completa

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SALIDAS = "/content/drive/MyDrive/tp_rl/outputs"

!python -m experimentos.correr_todo --dataset netflix --logs random mixto --base {SALIDAS}

Mounted at /content/drive

===== correr_todo 2026-11-23T13:36:35 | dataset netflix | logs ['random', 'mixto'] | semillas [0, 1, 2] | etapas ['baselines', 'dt', 'iql', 'reporte', 'comparacion'] =====
--- ya hecho, se saltea: [random] líneas base (/content/drive/MyDrive/tp_rl/outputs/baselines_netflix_random/sesiones.csv)
--- ya hecho, se saltea: [random] DT: entrenar semilla 0 (/content/drive/MyDrive/tp_rl/outputs/dt_netflix_random_seed0/modelo.pt)
--- ya hecho, se saltea: [random] DT: entrenar semilla 1 (/content/drive/MyDrive/tp_rl/outputs/dt_netflix_random_seed1/modelo.pt)
--- ya hecho, se saltea: [random] DT: entrenar semilla 2 (/content/drive/MyDrive/tp_rl/outputs/dt_netflix_random_seed2/modelo.pt)
--- ya hecho, se saltea: [random] DT: val dt_netflix_random_seed0 (/content/drive/MyDrive/tp_rl/outputs/dt_netflix_random_seed0/eval/val/mejor_retorno.json)
--- ya hecho, se saltea: [random] DT: val dt_netflix_random_seed1 (/content/drive/MyDrive/tp_rl/outputs/dt_netflix_random_seed1/eva